In [1]:
import math
import pandas as pd
from shapely.geometry import Point, Polygon
import os
import dask
from shapely.geometry import box
from pyproj import Transformer
import time

In [2]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import scipy
import zarr
#import dask
import re
import gc

In [3]:
#from xcube.core.store import find_data_store_extensions
#from xcube.core.store import get_data_store_params_schema

In [ ]:
import os
os.environ['http_proxy'] = '' 
os.environ['https_proxy'] = ''

In [5]:
from xcube.core.store import new_data_store

In [ ]:
sh_config = dict(client_id = '',
                 client_secret =  '',
                 oauth2_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect',
                 api_url = 'https://sh.dataspace.copernicus.eu',
                num_retries = 500)

In [7]:
#import geopy.distance

In [8]:
store = new_data_store('sentinelhub', **sh_config)

In [9]:
#get_data_store_params_schema("sentinelhub")

In [10]:
store.describe_data("sentinel-3-olci-l2")

In [12]:
test = store.open_data('sentinel-3-olci-l2',
                         variable_names = ['OTCI'],
                         bbox = [-4.478302,38.153457,-4.310417,38.270802],
                        upsampling = 'BILINEAR',
                        spatial_res = 0.01,
                        downsampling = 'BILINEAR',
                         crs = 'WGS84', #'EPSG:32632', #32632 #4326
                         time_range = ('2024-07-01', '2024-07-03'))

ValueError: cannot find collection name for dataset name 'sentinel-3-olci-l2'

In [11]:
store.describe_data("S3OLCI")

In [10]:
test = store.open_data('S3OLCI',
                         variable_names = ['OTCI'],
                         bbox = [-4.478302,38.153457,-4.310417,38.270802],
                        upsampling = 'BILINEAR',
                        spatial_res = 0.01,
                        downsampling = 'BILINEAR',
                         crs = 'WGS84', #'EPSG:32632', #32632 #4326
                         time_range = ('2024-07-01', '2024-07-03'))

/Net/Groups/BGI/scratch/dpabon/miniforge3/envs/sentinelhub_new/lib/python3.13/site-packages/xcube_sh/sentinelhub.py:254: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(dt, infer_datetime_format=True, utc=True)
/Net/Groups/BGI/scratch/dpabon/miniforge3/envs/sentinelhub_new/lib/python3.13/site-packages/xcube_sh/sentinelhub.py:317: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  pd.to_timedelta(max_timedelta)


In [11]:
test

<xarray.Dataset> Size: 2kB
Dimensions:    (time: 2, lat: 12, lon: 17, bnds: 2)
Coordinates:
  * lat        (lat) float64 96B 38.27 38.26 38.25 38.24 ... 38.18 38.17 38.16
  * lon        (lon) float64 136B -4.473 -4.463 -4.453 ... -4.333 -4.323 -4.313
  * time       (time) datetime64[ns] 16B 2024-07-01T10:16:13 2024-07-02T10:52:12
    time_bnds  (time, bnds) datetime64[ns] 32B dask.array<chunksize=(2, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    OTCI       (time, lat, lon) float32 2kB dask.array<chunksize=(1, 12, 17), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    title:                   S3OLCI Data Cube Subset
    history:                 [{'program': 'xcube_sh.chunkstore.SentinelHubChu...
    date_created:            2025-05-20T10:51:22.987591
    time_coverage_start:     2024-07-01T10:16:13+00:00
    time_coverage_end:       2024-07-02T10:52:12+00:00
    time_coverage_duration:  P1DT0H35M59S
    geospatial_lon_min:      -4.478302
    geospatial_lat_min:      38.153457
    geospatial_lon_max:      -4.308302
    geospatial_lat_max:      38.273457
    processing_level:        L1B

In [12]:
test.to_zarr('/Net/Groups/BGI/work_3/OEMC/oemc_sif/data/test_europe_CDSE_OTCI.zarr', mode = 'w')

Failed to fetch data from SentinelHub after 115.92606568336487 seconds and 500 retries
HTTP status code was 400
Failed to fetch data from SentinelHub after 116.65168309211731 seconds and 500 retries
HTTP status code was 400


SentinelHubError: 400 Client Error: Bad Request for url: https://sh.dataspace.copernicus.eu/api/v1/process

In [11]:
test['OTCI'].isel(time=0).plot()

Failed to fetch data from SentinelHub after 115.71284198760986 seconds and 500 retries
HTTP status code was 400


SentinelHubError: 400 Client Error: Bad Request for url: https://sh.dataspace.copernicus.eu/api/v1/process

In [5]:
from pyproj import CRS, Transformer
from shapely.geometry import Point
from shapely.ops import transform


def geodesic_point_buffer_lat_lon(lat, lon, km):
    # Azimuthal equidistant projection
    aeqd_proj = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat} +lon_0={lon} +x_0=0 +y_0=0")
    tfmr = Transformer.from_proj(aeqd_proj, aeqd_proj.geodetic_crs)
    buf = Point(0, 0).buffer(km * 1000)  # distance in metres
    arr = np.array(transform(tfmr.transform, buf).exterior.coords[:])
    
    return [arr[:,0].min(), arr[:,1].min(), arr[:,0].max(), arr[:,1].max()]

In [ ]:
def geo

In [32]:
## parameters:
site_id = 'JP-Mse'
coordinates_site = [36.0539,140.0269]
buffer_radius_km = 5
tmp_folder_zarr = '/Net/Groups/BGI/scratch/dpabon/tmp_folder_zarr/'
final_folder_zarr = '/Net/Groups/BGI/work_3/OEMC/oemc_towers/data/cubes/fluxnet_cubes/'
mode = 'w'
years = [2015, 2017, 2018, 2020, 2021, 2023, 2024, 2025]

bbox_geo = geodesic_point_buffer_lat_lon(coordinates_site[0], coordinates_site[1], buffer_radius_km)


In [33]:

def wgs84_to_utm(point_coordinates):
    """
    Convert a point from WGS84 to the most appropriate UTM CRS.

    Parameters:
    point_coordinates (tuple): A tuple containing (lat, lon) in WGS84.

    Returns:
    dict: A dictionary containing the UTM coordinates and the UTM zone string.
    """
    lat = point_coordinates[0]
    lon = point_coordinates[1]

    # Determine the UTM zone
    utm_zone = int((lon + 180) / 6) % 60 + 1
    hemisphere = 'north' if lat >= 0 else 'south'
    utm_crs = f"EPSG:326{utm_zone:02d}" if hemisphere == 'north' else f"EPSG:327{utm_zone:02d}"

    # Define the projections
    transformer = Transformer.from_crs("EPSG:4326", utm_crs)

    # Convert the point coordinates
    x, y = transformer.transform(lat, lon)

    # Return the UTM coordinates and the UTM CRS
    return {
        'utm_coords': (x, y),
        'utm_crs': utm_crs
    }


In [34]:
site_utm = wgs84_to_utm(coordinates_site)
site_utm

{'utm_coords': (412355.2233975863, 3990364.840175203), 'utm_crs': 'EPSG:32654'}

In [35]:
bbox = (site_utm['utm_coords'][0] - 5000, site_utm['utm_coords'][1] - 5000, site_utm['utm_coords'][0] + 5000, site_utm['utm_coords'][1] + 5000)
bbox

(407355.2233975863, 3985364.840175203, 417355.2233975863, 3995364.840175203)

In [36]:
site_utm['utm_crs']

'EPSG:32654'

In [30]:
dataset_s2 = store.open_data('S2L2A',
                         variable_names = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12', 'SCL'],
                         bbox = bbox,#(596820.448243452, 5654541.50582538, 606618.6245638039, 5664733.973906134), #(10.380851057590322, 51.03425592241329, 10.523548942409679, 51.12414373010549),#(596820.448243452, 5654541.50582538, 606618.6245638039, 5664733.973906134),
                        spatial_res = 20,
                        upsampling = 'BILINEAR',
                        downsampling = 'BILINEAR',
                         crs = site_utm['utm_crs'], #'EPSG:32632', #32632 #4326
                         time_range = ('2024-01-01', '2024-12-31'))

NameError: name 'bbox' is not defined

In [95]:
dataset_s2

<xarray.Dataset> Size: 646MB
Dimensions:    (time: 63, y: 500, x: 500, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 504B 2024-01-02T01:37:09 ... 2024-11-12T...
    time_bnds  (time, bnds) datetime64[ns] 1kB dask.array<chunksize=(63, 2), meta=np.ndarray>
  * x          (x) float64 4kB 4.074e+05 4.074e+05 ... 4.173e+05 4.173e+05
  * y          (y) float64 4kB 3.995e+06 3.995e+06 ... 3.985e+06 3.985e+06
Dimensions without coordinates: bnds
Data variables:
    B02        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B03        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B04        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B05        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B06        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B07        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B08        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B11        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B12        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    B8A        (time, y, x) float32 63MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    SCL        (time, y, x) uint8 16MB dask.array<chunksize=(1, 500, 500), meta=np.ndarray>
    crs        int64 8B ...
Attributes:
    Conventions:             CF-1.7
    title:                   S2L2A Data Cube Subset
    history:                 [{'program': 'xcube_sh.chunkstore.SentinelHubChu...
    date_created:            2024-11-13T11:18:30.536183
    time_coverage_start:     2024-01-02T01:37:00.778000+00:00
    time_coverage_end:       2024-11-12T01:37:17.431000+00:00
    time_coverage_duration:  P315DT0H0M16.653S
    processing_level:        L2A

In [96]:
dataset_s2.to_zarr('/Net/Groups/BGI/work_3/OEMC/oemc_towers/data/cubes/fluxnet_sentinelhub_tmp/JP-Mse/JP-Mse_2024.zarr')

In [75]:
# sites information
sites_location = pd.read_csv('/Net/Groups/BGI/work_3/OEMC/oemc_towers/data/sites_location.csv')
sites_location
sites_names = sites_location['site']

In [16]:
sites_location[sites_location['site'] == 'JP-Mse']

,site,long_name,CC,CC.1,lat,lon,altitude,IGBP,MAT,MAP
142,JP-Mse,Mase rice paddy field,NaN,CC-BY-4.0,36.0539,140.0269,NaN,CRO,NaN,NaN


In [8]:
sites_location[143:]

,site,long_name,CC,CC.1,lat,lon,altitude,IGBP,MAT,MAP
143,JP-SMF,Seto Mixed Forest Site,CC-BY-4.0,NaN,35.2617,137.0788,NaN,MF,NaN,NaN
144,JP-SwL,Suwa Lake,NaN,CC-BY-4.0,36.0466,138.1084,NaN,WAT,NaN,NaN
145,KR-CRK,Cheorwon Rice paddy,NaN,CC-BY-4.0,38.2013,127.2506,182.0,CRO,11.20,1180.9
146,MY-MLM,Maludam National Park,NaN,CC-BY-4.0,1.4536,111.1495,NaN,EBF,NaN,NaN
147,MY-PSO,Pasoh Forest Reserve (PSO),CC-BY-4.0,NaN,2.9730,102.3062,NaN,EBF,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
262,US-Wi8,Young hardwood clearcut (YHW),CC-BY-4.0,NaN,46.7223,-91.2524,348.0,DBF,NaN,NaN
263,US-Wi9,Young Jack pine (YJP),CC-BY-4.0,NaN,46.7385,-91.0746,350.0,ENF,NaN,NaN
264,US-Wkg,Walnut Gulch Kendall Grasslands,CC-BY-4.0,NaN,31.7365,-109.9419,1531.0,GRA,15.64,407.0
265,ZA-Kru,Skukuza,Tier 2,NaN,-25.0197,31.4969,359.0,SAV,21.90,547.0
